# Track Embedding Clustering — UMAP + HDBSCAN + Title Validation

**Approach:**
- UMAP reduces track embeddings (clustering signal) to low-dimensional Euclidean space
- HDBSCAN clusters in UMAP space
- Playlist title embeddings serve as a validation signal only — they are never used for clustering

**Pipeline:**
1. Load embeddings → diagnostic norm check
2. UMAP on track embeddings (compare 10, 15, 20 dims; default 15)
3. Visualize UMAP layout
4. HDBSCAN clustering (~50 clusters)
5. Manual title-vs-cluster spot check
6. Per-cluster average pairwise cosine similarity of title embeddings

# Setup

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_BASE           = "/content/drive/MyDrive/S2026/Spotify Playlist Data/Processed Data"
PROJECT_BASE         = "/content/drive/MyDrive/S2026/Spotify Playlist Data/Track vs Title"
EMBEDDINGS_PKL       = f"{DRIVE_BASE}/no_norm_embeddings.pkl"
CLUSTERS_OUTPUT_DIR  = f"{PROJECT_BASE}/umap_clusters"
REPO_DIR             = "/content/PlaylistClustering"

os.makedirs(CLUSTERS_OUTPUT_DIR, exist_ok=True)
print("Paths configured.")

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/siddmohanty111/PlaylistClustering.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

In [ ]:
%pip install -r {REPO_DIR}/requirements.txt
%pip install umap-learn hdbscan -q

In [ ]:
cluster_alts_path = os.path.join(REPO_DIR, "PlaylistRecsysUpgrade", "clustering", "cluster_alts.py")

with open(cluster_alts_path, 'r') as f:
    content = f.read()

# Replace the incorrect import statement
content = content.replace('from skfuzzy import cmeans, cmeans_predict', 'import skfuzzy as fuzz')

# Replace function calls
content = content.replace('cmeans(', 'fuzz.cmeans(')
content = content.replace('cmeans_predict(', 'fuzz.cmeans_predict(')

with open(cluster_alts_path, 'w') as f:
    f.write(content)

print(f"Successfully modified {cluster_alts_path} to correct skfuzzy imports.")

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "cluster_alts",
    os.path.join(REPO_DIR, "PlaylistRecsysUpgrade", "clustering", "cluster_alts.py"),
)
cluster_alts = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cluster_alts)

print("cluster_alts loaded.")

# Fix Pandas Version

In [ ]:
# # @title
# import subprocess, sys

# # Upgrade pandas to latest
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pandas", "-q"])

# # Restart runtime so the new pandas version is actually loaded
# import os
# os.kill(os.getpid(), 9)  # Forces a runtime restart — re-run all cells after this

# Imports

In [ ]:
import os
import pickle
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# Load Embeddings

In [ ]:
with open(EMBEDDINGS_PKL, 'rb') as f:
    data = pickle.load(f)

print("Full DataFrame shape:", data.shape)
print("Columns:", data.columns.tolist())
print(data.head())

In [ ]:
# Diagnostic: check whether embeddings are L2-normalized.
# SBERT and most sentence-transformer models output unit-norm vectors by default.
# If norms are all ~1.0, Euclidean distances concentrate around sqrt(2) regardless
# of dimensionality — explaining why PCA alone doesn't fix uniform memberships.
title_sample  = np.stack(data['title_embedding'].values[:200])
tracks_sample = np.stack(data['tracks_embedding'].values[:200])

title_norms  = np.linalg.norm(title_sample,  axis=1)
tracks_norms = np.linalg.norm(tracks_sample, axis=1)

print(f"Title  embedding norms — mean: {title_norms.mean():.4f}, std: {title_norms.std():.4f}")
print(f"Tracks embedding norms — mean: {tracks_norms.mean():.4f}, std: {tracks_norms.std():.4f}")
print()
if title_norms.std() < 0.01:
    print("⚠  Title embeddings appear L2-normalized (unit sphere).")
    print("   Euclidean distances will concentrate around √2 — this is why PCA alone")
    print("   does not fix uniform fuzzy memberships. UMAP is used below instead.")
else:
    print("Title embeddings are NOT unit-normalized — PCA alone may be sufficient.")

# UMAP — Track Embeddings

Fit UMAP on a 100k subsample of track embeddings, then batch-transform the full dataset.
We run three dimensionalities (10, 15, 20) so we can pick the best one before clustering.

In [ ]:
UMAP_FIT_SAMPLE = 100_000
TRANSFORM_BATCH = 10_000
UMAP_DIMS_TO_TRY = [10, 15, 20]
UMAP_DIM_DEFAULT = 15   # used downstream for clustering

rng = np.random.default_rng(42)

print("Stacking track embeddings...")
tracks_embeddings = np.stack(data['tracks_embedding'].values)
n_total = len(tracks_embeddings)
print(f"  Shape: {tracks_embeddings.shape}")

# Subsample for fitting
fit_idx = rng.choice(n_total, size=min(UMAP_FIT_SAMPLE, n_total), replace=False)
fit_data = tracks_embeddings[fit_idx]
print(f"  Fitting UMAP on {len(fit_idx):,} samples")

umap_results = {}   # dim -> (n_total, dim) array

for n_components in UMAP_DIMS_TO_TRY:
    print(f"\n── UMAP {n_components}d ──")
    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=15,
        min_dist=0.0,
        metric='cosine',
        low_memory=True,
        random_state=42,
        verbose=False,
    )
    reducer.fit(fit_data)
    print(f"  Fit complete. Transforming {n_total:,} points in batches...")

    parts = []
    for start in range(0, n_total, TRANSFORM_BATCH):
        batch = tracks_embeddings[start : start + TRANSFORM_BATCH]
        parts.append(reducer.transform(batch))
        print(f"  {min(start + TRANSFORM_BATCH, n_total):,} / {n_total:,}", end="\r")
    print()

    umap_results[n_components] = np.vstack(parts)
    print(f"  Done. Output shape: {umap_results[n_components].shape}")

del tracks_embeddings
print("\nAll UMAP reductions complete.")

# UMAP Dimension Selection Metrics

Three metrics help choose the best number of UMAP dimensions:

| Metric | What it measures | Want |
|---|---|---|
| **Trustworthiness** | Fraction of each point's true near-neighbours that are also near in UMAP space | High (→1) |
| **Continuity** | Fraction of UMAP near-neighbours that were also near in original space | High (→1) |
| **HDBSCAN noise fraction** | Fraction of points labelled as noise (−1) at a fixed `min_cluster_size` | Low |

Trustworthiness + continuity together form the **co-ranking** framework.
Higher is better for both. A large drop in either as dims decrease means that
dimensionality is too low to preserve neighbourhood structure faithfully.

In [ ]:
from sklearn.manifold import trustworthiness

# Run metrics on a small eval sample to keep it fast
EVAL_N = 5_000
eval_idx = rng.choice(n_total, size=EVAL_N, replace=False)

# Original space: use the already-stacked embeddings saved in data
original_sample = np.stack(data['tracks_embedding'].values[eval_idx])

rows = []
for dim, embedding in umap_results.items():
    umap_sample = embedding[eval_idx]

    tw = trustworthiness(original_sample, umap_sample, n_neighbors=15, metric='cosine')
    cont = trustworthiness(umap_sample, original_sample, n_neighbors=15, metric='euclidean')

    # Quick HDBSCAN probe
    probe = hdbscan.HDBSCAN(min_cluster_size=50, core_dist_n_jobs=-1)
    probe_labels = probe.fit_predict(umap_sample)
    noise_frac = (probe_labels == -1).mean()

    rows.append({
        'UMAP dims': dim,
        'Trustworthiness': round(tw, 4),
        'Continuity': round(cont, 4),
        'HDBSCAN noise %': round(noise_frac * 100, 2),
    })
    print(f"  {dim}d: trust={tw:.4f}  cont={cont:.4f}  noise={noise_frac:.2%}")

metrics_df = pd.DataFrame(rows).set_index('UMAP dims')
display(metrics_df)
print(f"\nDefault selected: {UMAP_DIM_DEFAULT}d  (change UMAP_DIM_DEFAULT above to override)")

# Visualize UMAP Layout (2D projection for plotting)

We run a separate 2D UMAP purely for visualization — the clustering uses
`UMAP_DIM_DEFAULT` dimensions, not 2.

In [ ]:
VIZ_SAMPLE = 20_000
viz_idx = rng.choice(n_total, size=VIZ_SAMPLE, replace=False)

print("Fitting 2D UMAP for visualization...")
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,   # slightly higher for cleaner scatter
    metric='cosine',
    low_memory=True,
    random_state=42,
    verbose=False,
)
umap_2d = reducer_2d.fit_transform(
    np.stack(data['tracks_embedding'].values[viz_idx])
)
print(f"  2D layout shape: {umap_2d.shape}")

plt.figure(figsize=(10, 8))
plt.scatter(umap_2d[:, 0], umap_2d[:, 1], s=3, alpha=0.4, linewidths=0)
plt.title(f"UMAP 2D — Track Embeddings (n={VIZ_SAMPLE:,} sample)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.axis('off')
plt.tight_layout()
plt.show()

# HDBSCAN Clustering on Track Embeddings

HDBSCAN is density-based: it finds clusters of arbitrary shape and marks low-density
points as noise (label `−1`). `min_cluster_size` is the primary knob — larger values
produce fewer, bigger clusters. We target ~50 clusters but HDBSCAN decides the exact
count; adjust `min_cluster_size` up/down to control this.

Note: `cluster_selection_method='eom'` (excess of mass) tends to produce more
clusters of varying size. Use `'leaf'` for more uniform cluster sizes.

In [ ]:
embedding_for_clustering = umap_results[UMAP_DIM_DEFAULT]

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=200,
    min_samples=10,
    cluster_selection_method='eom',
    core_dist_n_jobs=-1,
    prediction_data=True,           # needed for soft cluster membership if wanted later
)
print(f"Running HDBSCAN on {embedding_for_clustering.shape} embedding...")
cluster_labels = clusterer.fit_predict(embedding_for_clustering)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_frac  = (cluster_labels == -1).mean()
print(f"  Clusters found:  {n_clusters}")
print(f"  Noise points:    {noise_frac:.2%} of dataset")

# Attach labels to the main dataframe
data['track_cluster'] = cluster_labels

# Summary table
cluster_sizes = (
    data[data['track_cluster'] != -1]
    .groupby('track_cluster')
    .size()
    .rename('playlist_count')
    .sort_values(ascending=False)
    .reset_index()
)
print(f"\nCluster size summary (top 15):")
display(cluster_sizes.head(15))

In [ ]:
# Visualize clusters on the 2D layout
# Map the viz sample's cluster labels for colouring
viz_labels = cluster_labels[viz_idx]
noise_mask  = viz_labels == -1
cluster_mask = ~noise_mask

plt.figure(figsize=(12, 9))
plt.scatter(
    umap_2d[noise_mask, 0], umap_2d[noise_mask, 1],
    s=2, c='lightgrey', alpha=0.3, linewidths=0, label='Noise'
)
sc = plt.scatter(
    umap_2d[cluster_mask, 0], umap_2d[cluster_mask, 1],
    s=3, c=viz_labels[cluster_mask], cmap='turbo', alpha=0.6, linewidths=0
)
plt.colorbar(sc, label='Cluster ID')
plt.title(f"HDBSCAN on UMAP-{UMAP_DIM_DEFAULT}d Track Embeddings — {n_clusters} clusters")
plt.axis('off')
plt.tight_layout()
plt.show()

# Manual Spot-Check: Title vs Track Cluster Neighbours

Pick a few playlist titles below. For each, find the playlist in `data`,
then pull a random sample of other playlists in the same **track** cluster and
display their titles. This lets you judge by eye whether the track-based cluster
makes semantic sense relative to the title you chose as a reference.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Edit these titles to whatever you want to probe.
# The lookup is case-insensitive substring match, so partial names work.
PROBE_TITLES = [
    "chill vibes",
    "workout",
    "throwback",
]
NEIGHBOURS_TO_SHOW = 20   # how many cluster-mates to display per probe
# ───────────────────────────────────────────────────────────────────────────

# Drop noise points from lookup
clustered = data[data['track_cluster'] != -1].copy()

for probe in PROBE_TITLES:
    matches = clustered[clustered['playlist_title'].str.lower().str.contains(probe.lower())]
    if matches.empty:
        print(f'["{probe}"] — no exact match found, skipping.\n')
        continue

    # Pick the first match as the anchor
    anchor = matches.iloc[0]
    cluster_id = anchor['track_cluster']

    # Sample neighbours from the same track cluster (excluding the anchor itself)
    same_cluster = clustered[
        (clustered['track_cluster'] == cluster_id) &
        (clustered.index != anchor.name)
    ]
    sample = same_cluster.sample(min(NEIGHBOURS_TO_SHOW, len(same_cluster)), random_state=42)

    print(f'── Probe: "{anchor["playlist_title"]}"  →  Track cluster {cluster_id} '
          f'({len(same_cluster):,} playlists total) ──')
    display(sample[['playlist_title']].reset_index(drop=True))
    print()

# Per-Cluster Title Coherence — Average Pairwise Cosine Similarity

For each track cluster, sample up to `TITLE_SAMPLE_PER_CLUSTER` title embeddings
and compute their average pairwise cosine similarity. High similarity means the
playlists in that cluster tend to have semantically similar titles — a good sign
that track content and playlist intent are aligned. Low similarity means the cluster
groups musically similar tracks that come from thematically diverse playlists.

In [ ]:
TITLE_SAMPLE_PER_CLUSTER = 200   # max title embeddings to sample per cluster

clustered = data[data['track_cluster'] != -1].copy()
cluster_ids = sorted(clustered['track_cluster'].unique())

results = []
for cid in cluster_ids:
    group = clustered[clustered['track_cluster'] == cid]
    sample = group.sample(min(TITLE_SAMPLE_PER_CLUSTER, len(group)), random_state=42)

    # Stack and L2-normalise title embeddings (cosine sim = dot product after normalisation)
    title_vecs = np.stack(sample['title_embedding'].values)
    title_vecs = normalize(title_vecs, norm='l2')

    # Average pairwise cosine similarity = (sum of all dot products - n) / (n*(n-1))
    # Efficiently computed as: (||sum||^2 - n) / (n*(n-1))
    n = len(title_vecs)
    if n < 2:
        avg_sim = np.nan
    else:
        dot_matrix = title_vecs @ title_vecs.T   # (n, n)
        avg_sim = (dot_matrix.sum() - n) / (n * (n - 1))  # exclude self-similarities

    results.append({
        'cluster': cid,
        'n_playlists': len(group),
        'sample_size': n,
        'avg_pairwise_cosine_sim': round(float(avg_sim), 4),
    })

coherence_df = pd.DataFrame(results).sort_values('avg_pairwise_cosine_sim', ascending=False)
print("Top 10 clusters by title coherence (most thematically consistent):")
display(coherence_df.head(10))
print("\nBottom 10 clusters by title coherence (most thematically diverse):")
display(coherence_df.tail(10))

# Visualise
plt.figure(figsize=(14, 5))
sorted_for_plot = coherence_df.sort_values('cluster')
plt.bar(sorted_for_plot['cluster'].astype(str), sorted_for_plot['avg_pairwise_cosine_sim'], color='steelblue')
plt.axhline(coherence_df['avg_pairwise_cosine_sim'].mean(), color='red', linestyle='--', label='Mean')
plt.title('Per-Cluster Average Pairwise Cosine Similarity of Title Embeddings')
plt.xlabel('Track Cluster ID')
plt.ylabel('Avg Pairwise Cosine Sim')
plt.xticks(rotation=90)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nOverall mean: {coherence_df['avg_pairwise_cosine_sim'].mean():.4f}")
print(f"Std dev:      {coherence_df['avg_pairwise_cosine_sim'].std():.4f}")